# Augment Old `dot` files

The previous version of POGG worked with graphs that were represented like this in a `dot` file:

```
strict digraph  {
idDesk1 [node_type=entity_node, root=root];
yes;
true;
idGameRoom1;
idPlayer;
idDesk1 -> yes  [edge_type=property, label=idDesk1_prop_idImmovable];
idDesk1 -> true  [edge_type=property, label=idDesk1_prop_idUnknownObject];
idDesk1 -> idGameRoom1  [edge_type=relationship, label=isSouthOf];
idGameRoom1 -> true  [edge_type=property, label=idGameRoom1_prop_idUnknownObject];
idGameRoom1 -> idPlayer  [edge_type=relationship, label=isNorthOf];
}
```

The new version of POGG expects every node and edge to have a `lexicon_key` property. This puts the onus of deciding how node/edge names are regularized into lexicon keys on the user.

Ths notebook reads in the old subgraphs from the POGG experiments run for GP2 and adds the `lexicon_key` property to every node and edge using the same regularization that was used in the GP2 code. This should enable comparing experiment results from the new POGG with the old POGG to pinpoint any loss of coverage caused by the code overhaul.

In [6]:
import os
from pathlib import Path
import json
import networkx as nx
from perplexity_data_handling import regularize_node, regularize_edge

mode = "test"

if mode == "development":
    games = [
        "HealTheCar",
        "HealTheTrees",
        "Tutorial"
    ]
else:
    games = [
        "AtomicCity",
        "baby",
        "kidneykwest",
        "HealTheCave",
        "HealTheFlashback",
        "HealTheLake",
        "Scenario"
    ]

# Iterate over files in directory
for game in games:
    old_graphs_dir = "/Users/lizcconrad/Documents/PhD/POGG/POGG_project/POGG_data/{}/{}/subgraphs/dot/".format(mode, game)
    new_graphs_dir = "/Users/lizcconrad/Documents/PhD/POGG/data_handling/perplexity/{}/graph_jsons/{}/subgraphs/dot/".format(mode, game)
    # make new_graphs_dir
    Path(new_graphs_dir).mkdir(parents=True, exist_ok=True)

    for root, dirs, files in os.walk(old_graphs_dir):
        for subgraph_name in files:
            if subgraph_name.endswith(".dot"):
                subgraph_json = {'nodes': {}, 'edges': []}
                with open(os.path.join(root, subgraph_name), "r") as f:
                    graph = nx.drawing.nx_pydot.read_dot(f)

                    for n in graph.nodes:
                        subgraph_json['nodes'][n] = {
                            "lexicon_key": regularize_node(n),
                            "node_properties": graph.nodes[n]
                        }

                    for e in graph.edges.data():
                        parent = e[0]
                        child = e[1]
                        edge_data = e[2]
                        edge_name = edge_data['label']
                        # leaves only the other properties
                        del edge_data['label']

                        subgraph_json['edges'].append({
                            "edge_name": edge_name,
                            "parent_node": parent,
                            "child_node": child,
                            "lexicon_key": regularize_edge(edge_name),
                            "edge_properties": edge_data
                        })

                new_filename = subgraph_name.replace(".dot", ".json")
                with open(os.path.join(new_graphs_dir, new_filename), "w") as graph_json_file:
                    graph_json_file.write(json.dumps(subgraph_json, indent=4))
